# 🎥 Análise de pose no vídeo de fisioterapia

Módulo do sistema de monitoramento multimodal de pacientes.

Usa **MediaPipe Pose** para estimar as articulações frame-a-frame, calcula o **ângulo do joelho**
(quadril–joelho–tornozelo) e sinaliza como anomalia quando o ângulo fica fora de uma faixa esperada
por vários frames seguidos (ex: agachamento mal executado). Gera um relatório com os **timestamps**.

> Vídeo gravado por nós / de licença livre — não é de paciente real.

## 1. Dependências

In [ ]:
!pip install -q mediapipe opencv-python

## 2. Selecionar o vídeo

No Colab, faz upload do arquivo; localmente, usa um vídeo em `data/video/`.

In [ ]:
import os

VIDEO_PATH = None
try:
    from google.colab import files  # type: ignore
    print("Faça upload do vídeo (mp4/mov)...")
    enviado = files.upload()
    VIDEO_PATH = next(iter(enviado))
except Exception:
    # fallback local: ajuste para o nome do seu arquivo
    for nome in ["../data/video/exemplo.mp4", "../data/video/agachamento.mp4"]:
        if os.path.exists(nome):
            VIDEO_PATH = nome
            break

assert VIDEO_PATH, "Nenhum vídeo encontrado — faça upload ou coloque um arquivo em data/video/"
print("Vídeo:", VIDEO_PATH)

## 3. Função de ângulo entre três pontos

In [ ]:
import numpy as np

def angulo(a, b, c):
    """Ângulo (graus) em b, formado pelos pontos a-b-c."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc = a - b, c - b
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-9)
    return float(np.degrees(np.arccos(np.clip(cos, -1.0, 1.0))))

## 4. Estimativa de pose frame-a-frame

Para cada frame, extrai os landmarks e calcula o ângulo do joelho direito (quadril–joelho–tornozelo).

In [ ]:
import cv2
import mediapipe as mp
import pandas as pd

mp_pose = mp.solutions.pose
L = mp_pose.PoseLandmark

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
registros = []

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        res = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if res.pose_landmarks:
            lm = res.pose_landmarks.landmark
            def pt(i):
                return (lm[i].x, lm[i].y)
            ang = angulo(pt(L.RIGHT_HIP), pt(L.RIGHT_KNEE), pt(L.RIGHT_ANKLE))
            registros.append({"frame": frame_idx, "t_seg": frame_idx / fps, "angulo_joelho": ang})
        frame_idx += 1

cap.release()
df = pd.DataFrame(registros)
print(f"{len(df)} frames com pose detectada | fps={fps:.1f}")
df.head()

## 5. Regra de anomalia

Marca como anômalo o ângulo fora da faixa esperada por **pelo menos N frames consecutivos**
(evita disparo por ruído de um frame isolado).

In [ ]:
ANG_MIN, ANG_MAX = 70, 170   # faixa esperada do joelho (ajuste conforme o exercício)
MIN_FRAMES = 5               # nº mínimo de frames seguidos fora da faixa

df["fora_faixa"] = (df["angulo_joelho"] < ANG_MIN) | (df["angulo_joelho"] > ANG_MAX)

# agrupa sequências consecutivas de frames fora da faixa (posições inteiras no DataFrame)
fora = df["fora_faixa"].tolist()
anomalias = []  # lista de (pos_inicio, pos_fim) inclusivas
ini = None
for i, f in enumerate(fora):
    if f and ini is None:
        ini = i
    elif not f and ini is not None:
        if i - ini >= MIN_FRAMES:
            anomalias.append((ini, i - 1))
        ini = None
if ini is not None and len(fora) - ini >= MIN_FRAMES:
    anomalias.append((ini, len(fora) - 1))

print(f"{len(anomalias)} janela(s) de anomalia detectada(s)")

## 6. Relatório de timestamps

In [ ]:
relatorio = []
for a, b in anomalias:
    seg = df.iloc[a:b + 1]
    relatorio.append({
        "inicio_seg": round(float(df.iloc[a]["t_seg"]), 2),
        "fim_seg": round(float(df.iloc[b]["t_seg"]), 2),
        "articulacao": "joelho_direito",
        "angulo_min": round(float(seg["angulo_joelho"].min()), 1),
        "angulo_max": round(float(seg["angulo_joelho"].max()), 1),
    })

rel_df = pd.DataFrame(relatorio)
rel_df

## 7. Visualização

Ângulo do joelho ao longo do tempo, com a faixa esperada e as janelas anômalas destacadas.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(13, 4))
plt.plot(df["t_seg"], df["angulo_joelho"], lw=1, color="steelblue", label="ângulo do joelho")
plt.axhspan(ANG_MIN, ANG_MAX, color="green", alpha=0.08, label="faixa esperada")
for a, b in anomalias:
    plt.axvspan(df.iloc[a]["t_seg"], df.iloc[b]["t_seg"], color="red", alpha=0.2)
plt.xlabel("tempo (s)")
plt.ylabel("ângulo (graus)")
plt.title("Ângulo do joelho e janelas de anomalia")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Saída padronizada (para a integração)

In [ ]:
def anomalias_video(rel_df):
    """Lista de anomalias de movimento no formato consumido pelo dashboard."""
    return [
        {
            "timestamp": f"{r.inicio_seg:.2f}s–{r.fim_seg:.2f}s",
            "tipo": "movimento",
            "articulacao": r.articulacao,
            "detalhe": f"ângulo {r.angulo_min}–{r.angulo_max}° fora da faixa {ANG_MIN}–{ANG_MAX}°",
        }
        for r in rel_df.itertuples()
    ]

alertas = anomalias_video(rel_df) if len(rel_df) else []
print(f"{len(alertas)} alertas de movimento")
for a in alertas:
    print(a)

## 9. Conclusão

- O **MediaPipe Pose** extrai as articulações sem treino de modelo.
- A regra de **ângulo fora da faixa por N frames** sinaliza execução incorreta do exercício.
- Ajuste `ANG_MIN/ANG_MAX/MIN_FRAMES` conforme o exercício gravado.

A função `anomalias_video()` entrega os alertas no formato do alerta consolidado da equipe.